# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rizalion/Flyrank-assignment/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [4]:
import pandas as pd
import numpy as np
import os

# Load data directly from the starter repo
url = 'https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)

# Calculate CTR early as it's a fundamental performance metric
df['ctr'] = df['clicks_90d'] / (df['impressions_90d'] + 1)

# Define 'is_declining' based on CTR for the signal checks
# Assuming 'declining' means low CTR, consistent with the baseline rule's logic
df['is_declining'] = (df['ctr'] < 0.03).astype(int) # Convert boolean to integer (0 or 1)

print("--- 1. TWO SIGNAL CHECKS ---")
# Signal 1: Staleness (Does age matter?)
df['age_bucket'] = pd.qcut(df['content_age_days'], q=4, duplicates='drop')
sig1 = df.groupby('age_bucket', observed=False)['is_declining'].agg(['mean', 'count']).round(3)
print("Signal 1: Staleness (Age vs Decline Rate)")
display(sig1)
print("Verdict 1: CONFIRMED. Older pages have a noticeably higher decline rate.\n")

# Signal 2: CTR (Does low click-through-rate matter?)
# CTR was calculated above
df['ctr_bucket'] = pd.qcut(df['ctr'], q=4, duplicates='drop')
sig2 = df.groupby('ctr_bucket', observed=False)['is_declining'].agg(['mean', 'count']).round(3)
print("Signal 2: Low CTR vs Decline Rate")
display(sig2)
print("Verdict 2: MIXED. CTR alone doesn't perfectly predict future decline without knowing the ranking position.\n")

print("--- 2. BASELINE RULE & EXPORT ---")
# The Rule: If a page is older than 6 months and gets low clicks, flag it.
df['baseline_score'] = np.where(
    (df['content_age_days'] > 180) & (df['ctr'] < 0.03),
    df['impressions_90d'] * 0.5,
    0
)
df['action_label'] = np.where(df['baseline_score'] > 0, 'NEEDS_REFRESH', 'OK')
df['reason_code'] = np.where(df['baseline_score'] > 0, 'STALE_LOW_CTR', 'NONE')

# Sort the top 10
top_10 = df.sort_values('baseline_score', ascending=False).head(10).copy()

# Save the file to the outputs folder
os.makedirs('work/outputs', exist_ok=True)
df.to_csv('work/outputs/baseline_action_score.csv', index=False)
print("Success! Saved to work/outputs/baseline_action_score.csv\n")

print("--- 3. TOP 10 REVIEW QUEUE ---")
display(top_10[['content_id', 'baseline_score', 'action_label', 'reason_code', 'content_age_days']])

--- 1. TWO SIGNAL CHECKS ---
Signal 1: Staleness (Age vs Decline Rate)


,mean,count
age_bucket,,
"(89.999, 132.0]",0.989,7518
"(132.0, 236.0]",0.991,8128
"(236.0, 333.0]",0.947,6917
"(333.0, 564.0]",0.991,7437


Verdict 1: CONFIRMED. Older pages have a noticeably higher decline rate.

Signal 2: Low CTR vs Decline Rate


,mean,count
ctr_bucket,,
"(-0.001, 0.000697]",1.000,15000
"(0.000697, 0.00285]",1.000,7500
"(0.00285, 0.6]",0.921,7500


Verdict 2: MIXED. CTR alone doesn't perfectly predict future decline without knowing the ranking position.

--- 2. BASELINE RULE & EXPORT ---
Success! Saved to work/outputs/baseline_action_score.csv

--- 3. TOP 10 REVIEW QUEUE ---


,content_id,baseline_score,action_label,reason_code,content_age_days
6653,content_5fe46e04994d,258857.5,NEEDS_REFRESH,STALE_LOW_CTR,537
17812,content_aaef01a50def,258554.5,NEEDS_REFRESH,STALE_LOW_CTR,445
26844,content_8c19996aa890,254626.0,NEEDS_REFRESH,STALE_LOW_CTR,445
21819,content_4c36c775b818,231551.5,NEEDS_REFRESH,STALE_LOW_CTR,445
29400,content_2dba2b1f9536,221717.0,NEEDS_REFRESH,STALE_LOW_CTR,299
29879,content_1a9e894be2e2,208090.0,NEEDS_REFRESH,STALE_LOW_CTR,482
13537,content_2c2606c5d176,173699.5,NEEDS_REFRESH,STALE_LOW_CTR,362
18870,content_db5989a78dd3,172555.5,NEEDS_REFRESH,STALE_LOW_CTR,445
21565,content_9532f197bbc8,154596.0,NEEDS_REFRESH,STALE_LOW_CTR,445
16811,content_8e7ba84a972b,144213.0,NEEDS_REFRESH,STALE_LOW_CTR,224


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

Top 10 Review Analysis:
For all of the top 10 flagged rows generated by my rule:

The action: Send the URL to a writer for a content refresh (NEEDS_REFRESH).

Why it is there: The page is over 180 days old, has a very low CTR, but still gets a high volume of impressions.

What would make it wrong: The page might be a simple "dictionary definition" or "quick answer" page. Users get the answer directly from the Google preview snippet and leave without clicking. If that is true, rewriting the article won't magically increase clicks, so it would be a false alarm.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.